In [ ]:
import os, sys, time, math, pickle, shutil
from dataclasses import dataclass, field, asdict
from typing import Tuple, Optional, Callable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# ---------- reproducibility ----------
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- paper plotting style ----------
plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 200,
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 10, "axes.grid": True, "grid.alpha": 0.3,
    "legend.frameon": False, "figure.autolayout": False,
})
C_LEARN, C_RANDOM, C_GOOD, C_ACC = "#1A56A0", "#C05000", "#1D7A4F", "#7E3C8F"

# HYPERPARAMETER CONFIGURATION
DATASET_NAME   = "fashionmnist"        # Options: "mnist", "fashionmnist", "kmnist", "animeface"
DATA_ROOT      = "./data"       # Dataset root directory
BATCH_SIZE     = 128            # Batch size
EPOCHS         = 500             # Number of training epochs
LR             = 1e-3           # Learning rate
EMBED_DIM      = 128            # Embedding dimension
SNAPSHOT_EVERY = 5              # Save visualization snapshots every N epochs
OUT_DIR        = "./outputs"    # Output directory for checkpoints & figures

# Fixed structural parameters for ~30% visible in 7 steps
BUDGET         = 7              # 7 steps
GRID_SIZE      = 5              # 5x5 = 25 patches (7/25 = 28% â‰ˆ 30%)

# FIXED Loss coefficients (prevents critic dominance + entropy collapse)
RECON_COEF     = 1.0            # Reconstruction weight
ACTOR_COEF     = 0.1            # Policy gradient weight (was 0.01)
CRITIC_COEF    = 0.5            # Value loss weight
ENT_COEF_START = 0.05           # Initial entropy bonus (was 0.01)
ENT_COEF_END   = 0.01           # Final entropy bonus (was 0.001)
GRAD_CLIP      = 1.0            # Gradient clipping
GAE_LAMBDA     = 0.95           # GAE smoothing factor
VALUE_CLIP     = 2.0            # Clip value targets to prevent explosion

os.makedirs(OUT_DIR, exist_ok=True)

os.makedirs(OUT_DIR, exist_ok=True)

# ---------- dataset presets ----------
DATASET_PRESETS = {
    "mnist":       dict(image_size=40, patch_size=8,  channels=1,
                        mean=(0.1307,), std=(0.3081,), loader_fn="load_mnist"),
    "fashionmnist":dict(image_size=40, patch_size=8,  channels=1,
                        mean=(0.2860,), std=(0.3530,), loader_fn="load_fashionmnist"),
    "kmnist":      dict(image_size=40, patch_size=8,  channels=1,
                        mean=(0.1918,), std=(0.3483,), loader_fn="load_kmnist"),
    "animeface":   dict(image_size=50, patch_size=10, channels=3,
                        mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5), loader_fn="load_animeface"),
}

@dataclass
class Config:
    dataset_name: str
    image_size: int
    patch_size: int
    channels: int
    mean: tuple
    std: tuple
    batch_size: int
    budget: int
    grid_size: int = GRID_SIZE
    embed_dim: int = EMBED_DIM
    num_patches: int = field(init=False)
    visible_ratio: float = field(init=False)
    def __post_init__(self):
        assert self.image_size % self.patch_size == 0
        assert self.image_size // self.patch_size == self.grid_size
        self.num_patches = self.grid_size ** 2
        self.visible_ratio = self.budget / self.num_patches
        assert self.budget <= self.num_patches

def build_config(name): 
    p = DATASET_PRESETS[name]
    return Config(name, p["image_size"], p["patch_size"], p["channels"],
                  p["mean"], p["std"], BATCH_SIZE, BUDGET)

cfg = build_config(DATASET_NAME)

print("\n" + "="*70)
print(" ActiveDiffExplore â€” Hyperparameter Setup")
print("="*70)
print(f"[Config] {cfg.dataset_name} | image={cfg.image_size}Â² | patch={cfg.patch_size}Â² "
      f"| grid={cfg.grid_size}Ã—{cfg.grid_size} ({cfg.num_patches} patches) | "
      f"budget K={cfg.budget} | visibleâ‰ˆ{cfg.visible_ratio*100:.1f}%")
print("="*70 + "\n")



In [ ]:
def _build_transform(cfg):
    return T.Compose([
        T.Resize((cfg.image_size, cfg.image_size)),
        T.ToTensor(),
        T.Normalize(cfg.mean, cfg.std)
    ])

def load_mnist(cfg, root):
    tf = _build_transform(cfg)
    tr = torchvision.datasets.MNIST(root, train=True,  download=True, transform=tf)
    va = torchvision.datasets.MNIST(root, train=False, download=True, transform=tf)
    return tr, va

def load_fashionmnist(cfg, root):
    tf = _build_transform(cfg)
    tr = torchvision.datasets.FashionMNIST(root, train=True,  download=True, transform=tf)
    va = torchvision.datasets.FashionMNIST(root, train=False, download=True, transform=tf)
    return tr, va

def load_kmnist(cfg, root):
    tf = _build_transform(cfg)
    tr = torchvision.datasets.KMNIST(root, train=True,  download=True, transform=tf)
    va = torchvision.datasets.KMNIST(root, train=False, download=True, transform=tf)
    return tr, va

class _FlatImageFolder(Dataset):
    """Loads every image recursively under `root` (for the anime-face dataset)."""
    def __init__(self, root, transform):
        from PIL import Image
        self.transform = transform
        exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
        self.paths = []
        for dp, _, fns in os.walk(root):
            for f in fns:
                if f.lower().endswith(exts):
                    self.paths.append(os.path.join(dp, f))
        assert len(self.paths) > 0, f"No images found under {root}"
        print(f"  [FlatImageFolder] found {len(self.paths)} images under {root}")
        self._PIL = Image
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = self._PIL.open(self.paths[i]).convert("RGB")
        return self.transform(img), 0

def load_animeface(cfg, root):
    tf = _build_transform(cfg)
    full = os.path.join(root, "animeface")
    if not os.path.isdir(full): full = root
    ds = _FlatImageFolder(full, tf)
    n = len(ds); n_train = int(0.9 * n)
    g = torch.Generator().manual_seed(SEED)
    tr, va = torch.utils.data.random_split(ds, [n_train, n-n_train], generator=g)
    return tr, va

_LOADER_REGISTRY = {
    "load_mnist": load_mnist, "load_fashionmnist": load_fashionmnist,
    "load_kmnist": load_kmnist, "load_animeface": load_animeface,
}

def get_dataloaders(cfg, root):
    fn = _LOADER_REGISTRY[DATASET_PRESETS[cfg.dataset_name]["loader_fn"]]
    tr, va = fn(cfg, root)
    return (DataLoader(tr, batch_size=cfg.batch_size, shuffle=True,  num_workers=2,
                       pin_memory=True, drop_last=True),
            DataLoader(va, batch_size=cfg.batch_size, shuffle=False, num_workers=2,
                       pin_memory=True, drop_last=True))

train_loader, val_loader = get_dataloaders(cfg, DATA_ROOT)
sample_images, sample_labels = next(iter(train_loader))
print(f"[Data] train batches={len(train_loader)} | val batches={len(val_loader)} | "
      f"batch={tuple(sample_images.shape)}")



In [ ]:
def image_to_patch_grid(images, cfg):
    B, C, H, W = images.shape
    P, G = cfg.patch_size, cfg.grid_size
    p = images.unfold(2, P, P).unfold(3, P, P)
    p = p.permute(0, 2, 3, 1, 4, 5).contiguous().view(B, G*G, C, P, P)
    return p

def extract_single_patch(images, row, col, cfg):
    P = cfg.patch_size; B = images.size(0)
    out = torch.empty(B, cfg.channels, P, P, device=images.device, dtype=images.dtype)
    for b in range(B):
        r, c = row[b].item() * P, col[b].item() * P
        out[b] = images[b, :, r:r+P, c:c+P]
    return out

def action_to_coords(action, cfg):
    return action // cfg.grid_size, action % cfg.grid_size

def make_initial_mask(B, cfg, device=DEVICE):
    return torch.zeros(B, cfg.num_patches, dtype=torch.bool, device=device)

def update_mask(mask, action):
    m = mask.clone(); m.scatter_(1, action.unsqueeze(1), True); return m

def apply_mask_to_image(images, mask, cfg, fill=0.0):
    B, C, H, W = images.shape; P, G = cfg.patch_size, cfg.grid_size
    canvas = torch.full_like(images, fill)
    mg = mask.view(B, G, G)
    for b in range(B):
        for r in range(G):
            for c in range(G):
                if mg[b, r, c]:
                    canvas[b, :, r*P:(r+1)*P, c*P:(c+1)*P] = \
                        images[b, :, r*P:(r+1)*P, c*P:(c+1)*P]
    return canvas

def denormalize(images, cfg):
    m = torch.tensor(cfg.mean, device=images.device).view(1, -1, 1, 1)
    s = torch.tensor(cfg.std,  device=images.device).view(1, -1, 1, 1)
    return torch.clamp(images * s + m, 0, 1)

# --- sanity check ---
patches = image_to_patch_grid(sample_images, cfg)
B, N, C, P, _ = patches.shape; G = cfg.grid_size
rebuilt = patches.view(B, G, G, C, P, P).permute(0, 3, 1, 4, 2, 5).contiguous().view(B, C, G*P, G*P)
err = (rebuilt - sample_images).abs().max().item()
print(f"[Sanity] patch round-trip max error = {err:.2e}  ({'PASS' if err < 1e-5 else 'FAIL'})")



In [ ]:
def plot_samples_and_grid(images, labels, cfg, n_show=8, save_path=None):
    n_show = min(n_show, images.size(0))
    imgs = denormalize(images[:n_show], cfg).cpu()
    fig, axes = plt.subplots(1, n_show, figsize=(2*n_show, 2.4), dpi=150)
    if n_show == 1: axes = [axes]
    cmap = "gray" if cfg.channels == 1 else None
    for i in range(n_show):
        ax = axes[i]
        img = imgs[i].permute(1, 2, 0).squeeze().numpy()
        ax.imshow(img, cmap=cmap)
        for g in range(1, cfg.grid_size):
            ax.axhline(g*cfg.patch_size - 0.5, color="red", ls="--", lw=0.8)
            ax.axvline(g*cfg.patch_size - 0.5, color="red", ls="--", lw=0.8)
        for r in range(cfg.grid_size):
            for c in range(cfg.grid_size):
                idx = r * cfg.grid_size + c
                ax.text(c*cfg.patch_size + cfg.patch_size/2,
                        r*cfg.patch_size + cfg.patch_size/2,
                        str(idx), color="lime", fontsize=7,
                        weight="bold", ha="center", va="center")
        ax.set_title(f"y={int(labels[i])}" if labels is not None else "", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"{cfg.dataset_name.upper()} â€” patch grid "
                 f"({cfg.grid_size}Ã—{cfg.grid_size} = {cfg.num_patches} patches, "
                 f"K={cfg.budget} visible â‰ˆ {cfg.visible_ratio*100:.0f}%)",
                 fontsize=12, weight="bold")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, bbox_inches="tight")
    plt.show()

plot_samples_and_grid(sample_images, sample_labels, cfg, n_show=8,
                      save_path=os.path.join(OUT_DIR, "fig_samples_and_grid.png"))



In [ ]:
class CNNPatchEncoder(nn.Module):
    def __init__(self, cfg, embed_dim=EMBED_DIM):
        super().__init__()
        C, P = cfg.channels, cfg.patch_size
        self.net = nn.Sequential(
            nn.Conv2d(C, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((P//4, P//4)),
            nn.Flatten(),
            nn.Linear(64*(P//4)*(P//4), embed_dim)
        )
    def forward(self, x): return self.net(x)

class TransformerSceneMemory(nn.Module):
    def __init__(self, cfg, embed_dim=EMBED_DIM, num_heads=4, num_layers=2):
        super().__init__()
        G, K = cfg.grid_size, cfg.budget
        self.row_emb  = nn.Embedding(G, embed_dim)
        self.col_emb  = nn.Embedding(G, embed_dim)
        self.time_emb = nn.Embedding(K, embed_dim)
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads,
                                           dim_feedforward=embed_dim*4,
                                           batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
    def forward(self, embs, coords, steps):
        t = embs + self.row_emb(coords[...,0]) + self.col_emb(coords[...,1]) + self.time_emb(steps)
        return self.transformer(t)

class ActiveExplorationPolicy(nn.Module):
    def __init__(self, cfg, embed_dim=EMBED_DIM):
        super().__init__()
        N = cfg.num_patches
        self.pooler = nn.AdaptiveAvgPool1d(1)
        self.actor  = nn.Sequential(nn.Linear(embed_dim+1, 256), nn.ReLU(inplace=True),
                                    nn.Linear(256, 256), nn.ReLU(inplace=True),
                                    nn.Linear(256, N))
        self.critic = nn.Sequential(nn.Linear(embed_dim+1, 128), nn.ReLU(inplace=True),
                                    nn.Linear(128, 1))
    def forward(self, mem, visited, budget_frac):
        g = self.pooler(mem.permute(0, 2, 1)).squeeze(-1)
        ctx = torch.cat([g, budget_frac], dim=-1)
        logits = self.actor(ctx).masked_fill(visited, float("-inf"))
        value  = self.critic(ctx)
        return logits, value
    @staticmethod
    def sample_action(logits, deterministic=False):
        probs = F.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)
        a = torch.argmax(probs, dim=-1) if deterministic else dist.sample()
        return a, dist.log_prob(a), dist.entropy()

class ReconstructionDecoder(nn.Module):
    """FIXED: Uses interpolation-based upsampling to handle any image size correctly."""
    def __init__(self, cfg, embed_dim=EMBED_DIM):
        super().__init__()
        self.cfg = cfg
        self.pooler = nn.AdaptiveAvgPool1d(1)
        self.seed_size = 4
        self.proj = nn.Linear(embed_dim, 256 * self.seed_size * self.seed_size)
        
        # Decoder blocks with interpolation upsampling
        self.dec_blocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(256, 128, 3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(inplace=True)
            ),
            nn.Sequential(
                nn.Conv2d(128, 64, 3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True)
            ),
            nn.Sequential(
                nn.Conv2d(64, 32, 3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True)
            ),
        ])
        
        # Final output layer
        self.final_conv = nn.Conv2d(32, cfg.channels + 1, 3, padding=1)
        
        # Target sizes for each upsampling step
        img_size = cfg.image_size
        self.upsample_targets = [
            img_size // 4,   # 10 for 40x40
            img_size // 2,   # 20 for 40x40
            img_size,        # 40 for 40x40
        ]

    def forward(self, mem):
        B = mem.size(0)
        g = self.pooler(mem.permute(0, 2, 1)).squeeze(-1)
        x = self.proj(g).view(B, 256, self.seed_size, self.seed_size)
        
        for block, target_size in zip(self.dec_blocks, self.upsample_targets):
            x = F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)
            x = block(x)
        
        out = self.final_conv(x)
        recon = torch.tanh(out[:, :self.cfg.channels])
        unc   = torch.sigmoid(out[:, self.cfg.channels:])
        return recon, unc

class ActiveDiffExploreModel(nn.Module):
    def __init__(self, cfg, embed_dim=EMBED_DIM):
        super().__init__()
        self.cfg = cfg
        self.encoder = CNNPatchEncoder(cfg, embed_dim)
        self.memory  = TransformerSceneMemory(cfg, embed_dim)
        self.policy  = ActiveExplorationPolicy(cfg, embed_dim)
        self.decoder = ReconstructionDecoder(cfg, embed_dim)

model = ActiveDiffExploreModel(cfg, EMBED_DIM).to(DEVICE)
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[Model] trainable params = {n_params:,}")



In [ ]:
def plot_architecture(cfg, save_path=None):
    fig, ax = plt.subplots(figsize=(14, 6), dpi=200)
    ax.set_xlim(0, 13); ax.set_ylim(0, 6.8); ax.axis("off")
    def box(x, y, w, h, txt, color="#E8F0FE", ec="#1A73E8"):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                    fc=color, ec=ec, lw=1.5))
        ax.text(x+w/2, y+h/2, txt, ha="center", va="center",
                fontsize=8, style="italic", color="#333")

    def arrow(x1, y1, x2, y2, txt=""):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="->", color="#555", lw=1.2))
        if txt: ax.text((x1+x2)/2, (y1+y2)/2+0.15, txt, fontsize=7, ha="center", color="#444")

    box(0.2, 4.8, 2.0, 1.2, f"Input Image\n{cfg.image_size}Ã—{cfg.image_size}Ã—{cfg.channels}", color="#FFF3D6", ec="#C05000")
    box(0.2, 2.6, 2.0, 1.2, "Patch Selector\n(grid tokens)", color="#FFE9E9", ec="#C03030")
    box(0.2, 0.4, 2.0, 1.2, f"Observed Patch\n{cfg.patch_size}Ã—{cfg.patch_size}Ã—{cfg.channels}", color="#FFF3D6", ec="#C05000")
    box(3.0, 0.4, 2.2, 1.2, "CNN Patch\nEncoder", color="#E6F0FA", ec="#1A56A0")
    box(3.0, 2.6, 2.2, 1.2, "Transformer Scene\nMemory\n(pos+time emb)", color="#E8F5E9", ec="#1D7A4F")
    box(6.0, 4.4, 2.4, 1.4, "PPO Actor-Critic\nPolicy\n(logits + value)", color="#F3E8FA", ec="#7E3C8F")
    box(6.0, 1.4, 2.4, 1.6, "Reconstruction\nDecoder\n(Interp+Conv)", color="#E8F5E9", ec="#1D7A4F")
    box(9.4, 4.4, 1.8, 1.4, "Action\n(next patch)", color="#FFE9E9", ec="#C03030")
    box(9.4, 1.4, 1.8, 1.6, "Recon +\nUncertainty", color="#FFF3D6", ec="#C05000")
    box(11.6, 3.0, 1.2, 1.6, "Loss\n(MSE+PPO)", color="#FCE4EC", ec="#C2185B")

    arrow(2.2, 5.4, 6.0, 5.0, "state")
    arrow(2.2, 3.2, 3.0, 3.2)
    arrow(2.2, 1.0, 3.0, 1.0)
    arrow(5.2, 1.0, 5.7, 1.0)
    arrow(5.7, 1.0, 6.0, 2.0)
    arrow(5.2, 3.2, 6.0, 4.6)
    arrow(5.2, 3.2, 6.0, 2.2)
    arrow(8.4, 5.1, 9.4, 5.1, "argmax/sample")
    arrow(8.4, 2.2, 9.4, 2.2)
    arrow(9.4, 5.1, 11.6, 4.0, "log_prob")
    arrow(9.4, 2.2, 11.6, 3.4, "MSE")
    arrow(2.2, 5.4, 0.2, 4.0)
    arrow(9.4, 5.1, 2.3, 3.8, "next patch (loop)")

    ax.text(6.5, 6.2, "ActiveDiffExplore â€” Architecture Overview",
            ha="center", fontsize=14, weight="bold")
    ax.text(6.5, 0.05, f"Budget K = {cfg.budget} steps  |  "
                       f"{cfg.num_patches} patches  |  "
                       f"~{cfg.visible_ratio*100:.0f}% visible",
            ha="center", fontsize=10, color="#555")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, bbox_inches="tight")
    plt.show()

plot_architecture(cfg, save_path=os.path.join(OUT_DIR, "fig_architecture.png"))



In [ ]:

# ---- Fixed hyperparameters ----
RECON_COEF     = 1.0
ACTOR_COEF     = 0.1       # Increased from 0.01
CRITIC_COEF    = 0.5       # Keep same but we'll normalize targets
ENT_COEF_START = 0.05      # Increased from 0.01 (prevent entropy collapse)
ENT_COEF_END   = 0.01      # Increased from 0.001
GRAD_CLIP      = 1.0
GAE_LAMBDA     = 0.95      # New: GAE smoothing factor
VALUE_CLIP     = 0.2       # New: Clip value targets to prevent explosion

# Running reward statistics for normalization
reward_ema_mean = 0.0
reward_ema_std  = 1.0

def run_episode(model, images_raw01, cfg, deterministic=False, random_start=False):
    """Run one episode of active patch selection."""
    B = images_raw01.size(0)
    visited = make_initial_mask(B, cfg)
    embs_h, coords_h, steps_h = [], [], []
    log_probs_h, entropies_h, values_h = [], [], []
    
    if random_start:
        action = torch.randint(0, cfg.num_patches, (B,), device=DEVICE)
    else:
        action = torch.zeros(B, dtype=torch.long, device=DEVICE)
    
    for t in range(cfg.budget):
        row, col = action_to_coords(action, cfg)
        p = extract_single_patch(images_raw01, row, col, cfg)
        embs_h.append(model.encoder(p).unsqueeze(1))
        coords_h.append(torch.stack([row, col], dim=-1).unsqueeze(1))
        steps_h.append(torch.full((B,1), t, dtype=torch.long, device=DEVICE))
        
        mem = model.memory(torch.cat(embs_h, 1), torch.cat(coords_h, 1), torch.cat(steps_h, 1))
        visited = update_mask(visited, action)
        
        if t < cfg.budget - 1:
            bf = torch.full((B,1), (cfg.budget - t) / cfg.budget, device=DEVICE)
            logits, value = model.policy(mem, visited, bf)
            action, log_prob, entropy = ActiveExplorationPolicy.sample_action(logits, deterministic)
            log_probs_h.append(log_prob)
            entropies_h.append(entropy)
            values_h.append(value.squeeze(-1))
    
    recon_pm1, unc = model.decoder(mem)
    recon01 = torch.clamp((recon_pm1 + 1) / 2, 0, 1)
    
    return dict(recon_pm1=recon_pm1, recon01=recon01, uncertainty=unc,
                target_pm1=images_raw01 * 2 - 1,
                log_probs=log_probs_h, entropies=entropies_h, values=values_h,
                coords_trace=[c.squeeze(1) for c in coords_h])

def compute_psnr(x, y):
    mse = F.mse_loss(x, y, reduction='none').mean(dim=[1,2,3])
    return (10 * torch.log10(1.0 / (mse + 1e-10))).mean().item()

def reward_from_psnr(recon, target):
    with torch.no_grad():
        mse = F.mse_loss(recon, target, reduction='none').mean(dim=[1,2,3])
        return 10 * torch.log10(1.0 / (mse + 1e-10))

def normalize_reward(reward, ema_mean, ema_std, beta=0.99):
    """Update EMA statistics and return normalized reward."""
    batch_mean = reward.mean().item()
    batch_std = reward.std().item() + 1e-6
    new_mean = beta * ema_mean + (1 - beta) * batch_mean
    new_std = beta * ema_std + (1 - beta) * batch_std
    normalized = (reward - new_mean) / (new_std + 1e-6)
    return normalized, new_mean, new_std

def compute_losses(out, cfg, ent_coef=ENT_COEF_START, reward_stats=None):
    """
    FIXED loss computation:
    1. Proper size matching for reconstruction
    2. Reward normalization (prevents critic explosion)
    3. GAE for advantage estimation (more stable)
    4. Value target clipping (prevents critic blowup)
    5. Stronger entropy regularization
    """
    global reward_ema_mean, reward_ema_std
    
    # ---- Reconstruction loss (with size safety) ----
    recon = out["recon_pm1"]
    target = out["target_pm1"]
    if recon.shape != target.shape:
        target = F.interpolate(target, size=recon.shape[2:], mode='bilinear', align_corners=False)
    recon_loss = F.mse_loss(recon, target)
    
    # ---- Normalize reward (KEY FIX) ----
    with torch.no_grad():
        rraw = reward_from_psnr(recon, target)
        rnorm, reward_ema_mean, reward_ema_std = normalize_reward(
            rraw, reward_ema_mean, reward_ema_std)
    
    # ---- GAE Advantage estimation ----
    n_steps = len(out["values"])
    if n_steps == 0:
        return RECON_COEF * recon_loss, dict(
            recon_loss=recon_loss.item(), actor_loss=0.0, critic_loss=0.0,
            mean_reward=rraw.mean().item(), entropy=0.0)
    
    # Compute TD residuals with normalized rewards
    values = out["values"]
    td_residuals = []
    for t in range(n_steps):
        if t == n_steps - 1:
            # Last step: bootstrap with normalized terminal reward
            next_val = rnorm
        else:
            next_val = values[t + 1]
        td_residuals.append(rnorm - values[t] + GAE_LAMBDA * (next_val - values[t]))
    
    # Compute GAE advantages
    advantages = []
    gae = torch.zeros_like(rnorm)
    for t in reversed(range(n_steps)):
        gae = td_residuals[t] + GAE_LAMBDA * gae
        advantages.append(gae)
    advantages = list(reversed(advantages))
    
    # Normalize advantages
    adv_stack = torch.stack(advantages)
    adv_normalized = (adv_stack - adv_stack.mean()) / (adv_stack.std() + 1e-6)
    
    # ---- Value targets with clipping (prevents critic explosion) ----
    value_targets = adv_normalized + torch.stack(values)
    # Clip value targets to prevent huge gradients
    value_targets = torch.clamp(value_targets, -VALUE_CLIP / CRITIC_COEF, VALUE_CLIP / CRITIC_COEF)
    
    # ---- Compute actor and critic losses ----
    a_terms = []
    c_terms = []
    for t in range(n_steps):
        # Actor: policy gradient with normalized advantage + entropy bonus
        a_terms.append(-out["log_probs"][t] * adv_normalized[t] 
                       - ent_coef * out["entropies"][t])
        # Critic: clipped value loss
        c_terms.append(F.mse_loss(values[t], value_targets[t]))
    
    al = torch.stack(a_terms).mean()
    cl = torch.stack(c_terms).mean()
    
    # Total loss
    total = RECON_COEF * recon_loss + ACTOR_COEF * al + CRITIC_COEF * cl
    
    me = torch.stack(out["entropies"]).mean().item()
    return total, dict(recon_loss=recon_loss.item(), actor_loss=al.item(),
                       critic_loss=cl.item(), mean_reward=rraw.mean().item(), entropy=me)

def ent_schedule(epoch, total, s=ENT_COEF_START, e=ENT_COEF_END):
    """Slower entropy decay."""
    return s + min(epoch / total, 1.0) * (e - s)

def train_one_epoch(model, opt, loader, cfg, epoch):
    model.train()
    agg = {k: 0.0 for k in ["total", "recon", "actor", "critic", "reward", "entropy", "psnr"]}
    nb = 0
    ec = ent_schedule(epoch, EPOCHS)
    
    for images, _ in loader:
        images = images.to(DEVICE)
        raw01 = denormalize(images, cfg)
        out = run_episode(model, raw01, cfg, deterministic=False, random_start=True)
        loss, st = compute_losses(out, cfg, ec)
        
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        
        with torch.no_grad():
            psnr = compute_psnr(out["recon01"], raw01)
        agg["total"] += loss.item()
        agg["psnr"] += psnr
        nb += 1
        for k, kk in [("recon", "recon_loss"), ("actor", "actor_loss"),
                      ("critic", "critic_loss"), ("reward", "mean_reward"), ("entropy", "entropy")]:
            agg[k] += st[kk]
    
    return {k: v / nb for k, v in agg.items()}

@torch.no_grad()
def evaluate(model, loader, cfg, max_batches=20):
    model.eval()
    ps, rs = 0., 0.
    nb = 0
    for i, (images, _) in enumerate(loader):
        if i >= max_batches:
            break
        images = images.to(DEVICE)
        raw01 = denormalize(images, cfg)
        out = run_episode(model, raw01, cfg, deterministic=True, random_start=True)
        ps += compute_psnr(out["recon01"], raw01)
        # Use size-safe comparison
        recon = out["recon_pm1"]
        target = out["target_pm1"]
        if recon.shape != target.shape:
            target = F.interpolate(target, size=recon.shape[2:], mode='bilinear', align_corners=False)
        rs += F.mse_loss(recon, target).item()
        nb += 1
    return {"psnr": ps / nb, "recon_loss": rs / nb}

@torch.no_grad()
def capture_snapshot(model, loader, cfg, n=4):
    model.eval()
    images, _ = next(iter(loader))
    images = images[:n].to(DEVICE)
    raw01 = denormalize(images, cfg)
    out = run_episode(model, raw01, cfg, deterministic=True, random_start=True)
    return dict(gt=raw01.cpu(), recon=out["recon01"].cpu(),
                uncertainty=out["uncertainty"].cpu(),
                coords_trace=[c.cpu() for c in out["coords_trace"]])

# ---- Test that shapes match ----
print("[Sanity Check] Testing loss computation...")
with torch.no_grad():
    test_imgs = sample_images[:4].to(DEVICE)
    test_raw = denormalize(test_imgs, cfg)
    test_out = run_episode(model, test_raw, cfg, deterministic=True, random_start=True)
    print(f"  recon shape: {test_out['recon_pm1'].shape}")
    print(f"  target shape: {test_out['target_pm1'].shape}")
    test_loss, test_stats = compute_losses(test_out, cfg)
    print(f"  test loss: {test_loss.item():.4f}")
    print(f"  recon_loss: {test_stats['recon_loss']:.4f}")
    print(f"  critic_loss: {test_stats['critic_loss']:.4f}")
    print(f"  entropy: {test_stats['entropy']:.4f}")
    print("[Sanity Check] PASSED âœ“\n")

In [ ]:
history = {k: [] for k in ["epoch", "total", "recon", "actor", "critic", "reward",
                           "entropy", "psnr", "val_psnr", "val_recon", "time"]}
snapshots = []
best_val_psnr = 0.0  # Track best model

print(f"[Train] {cfg.dataset_name} | K={cfg.budget} | epochs={EPOCHS} | "
      f"batches/epoch={len(train_loader)}\n")
print(f"[Save] Best model will be saved to: {os.path.join(OUT_DIR, 'best_model.pt')}")
print(f"[Save] Latest checkpoint: {os.path.join(OUT_DIR, 'latest.pt')}\n")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = train_one_epoch(model, optimizer, train_loader, cfg, epoch)
    va = evaluate(model, val_loader, cfg)
    dt = time.time() - t0
    
    # Log history
    history["epoch"].append(epoch)
    history["time"].append(dt)
    for k in ["total", "recon", "actor", "critic", "reward", "entropy", "psnr"]:
        history[k].append(tr[k])
    history["val_psnr"].append(va["psnr"])
    history["val_recon"].append(va["recon_loss"])
    
    # Print progress
    best_marker = ""
    if va["psnr"] > best_val_psnr:
        best_val_psnr = va["psnr"]
        best_marker = " â˜… BEST"
    
    print(f"  ep{epoch:03d}/{EPOCHS} | loss={tr['total']:.4f} | recon={tr['recon']:.4f} | "
          f"actor={tr['actor']:.4f} | critic={tr['critic']:.4f} | rew={tr['reward']:.2f}dB | "
          f"ent={tr['entropy']:.3f} | trPSNR={tr['psnr']:.2f} | vaPSNR={va['psnr']:.2f}{best_marker} | {dt:.1f}s")
    
    # ---- SAVE BEST MODEL ----
    if va["psnr"] > best_val_psnr or epoch == 1:
        best_val_psnr = va["psnr"]
        best_save = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_psnr": best_val_psnr,
            "config": asdict(cfg),
            "hyperparams": {
                "embed_dim": EMBED_DIM,
                "budget": BUDGET,
                "grid_size": GRID_SIZE,
                "recon_coef": RECON_COEF,
                "actor_coef": ACTOR_COEF,
                "critic_coef": CRITIC_COEF,
            }
        }
        torch.save(best_save, os.path.join(OUT_DIR, "best_model.pt"))
    
    # ---- SAVE LATEST CHECKPOINT (for resuming training) ----
    latest_save = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_psnr": best_val_psnr,
        "config": asdict(cfg),
        "hyperparams": {
            "embed_dim": EMBED_DIM,
            "budget": BUDGET,
            "grid_size": GRID_SIZE,
            "recon_coef": RECON_COEF,
            "actor_coef": ACTOR_COEF,
            "critic_coef": CRITIC_COEF,
        },
        "reward_ema_mean": reward_ema_mean,
        "reward_ema_std": reward_ema_std,
        "history": history,
    }
    torch.save(latest_save, os.path.join(OUT_DIR, "latest.pt"))
    
    # ---- SAVE SNAPSHOT VISUALIZATIONS ----
    if epoch % SNAPSHOT_EVERY == 0 or epoch == EPOCHS:
        s = capture_snapshot(model, val_loader, cfg)
        s["epoch"] = epoch
        snapshots.append(s)

# ---- FINAL SAVE ----
with open(os.path.join(OUT_DIR, "results.pkl"), "wb") as f:
    pickle.dump({"history": history, "snapshots": snapshots, "cfg": asdict(cfg)}, f)

print("\n" + "=" * 70)
print(f"[Train] Done!")
print(f"[Save] Best model (PSNR={best_val_psnr:.2f}): {os.path.join(OUT_DIR, 'best_model.pt')}")
print(f"[Save] Latest checkpoint: {os.path.join(OUT_DIR, 'latest.pt')}")
print(f"[Save] Results: {os.path.join(OUT_DIR, 'results.pkl')}")
print("=" * 70)

In [ ]:
def plot_training_curves(history, cfg, save_path=None):
    cfg_dict = asdict(cfg) if not isinstance(cfg, dict) else cfg
    fig, ax = plt.subplots(2, 3, figsize=(15, 8.5), dpi=200)
    ep = history["epoch"]
    
    ax[0,0].plot(ep, history["total"], "-",  color=C_LEARN, label="Total")
    ax[0,0].plot(ep, history["recon"], "--", color="#2E86AB", label="Recon")
    ax[0,0].set_title("Training Loss"); ax[0,0].set_xlabel("Epoch"); ax[0,0].set_ylabel("Loss"); ax[0,0].legend()
    
    ax[0,1].plot(ep, history["psnr"],    "-",  color=C_LEARN, label="Train")
    ax[0,1].plot(ep, history["val_psnr"],"-",  color=C_GOOD,  label="Val")
    ax[0,1].set_title("Reconstruction PSNR"); ax[0,1].set_xlabel("Epoch"); ax[0,1].set_ylabel("PSNR (dB)"); ax[0,1].legend()
    
    ax[0,2].plot(ep, history["critic"], "-", color=C_RANDOM)
    ax[0,2].set_title("Critic Loss"); ax[0,2].set_xlabel("Epoch"); ax[0,2].set_ylabel("Loss")
    
    ax[1,0].plot(ep, history["entropy"], "-", color=C_RANDOM)
    ax[1,0].axhline(np.log(cfg_dict["num_patches"]), color="red", ls="--", lw=1,
                    label=f"max entropy (ln {cfg_dict['num_patches']})")
    ax[1,0].set_title("Policy Entropy"); ax[1,0].set_xlabel("Epoch"); ax[1,0].set_ylabel("Entropy (nats)"); ax[1,0].legend()
    
    ax[1,1].plot(ep, history["actor"], "-", color=C_RANDOM)
    ax[1,1].set_title("Actor Loss"); ax[1,1].set_xlabel("Epoch"); ax[1,1].set_ylabel("Loss")
    
    ax[1,2].plot(ep, history["reward"], "-", color=C_GOOD)
    ax[1,2].set_title("Mean Terminal Reward (PSNR)"); ax[1,2].set_xlabel("Epoch"); ax[1,2].set_ylabel("dB")
    
    fig.suptitle(f"ActiveDiffExplore Training Curves â€” {cfg_dict['dataset_name'].upper()}  "
                 f"(K={cfg_dict['budget']}, visibleâ‰ˆ{cfg_dict['budget']/cfg_dict['num_patches']*100:.0f}%)",
                 fontsize=13, weight="bold")
    plt.tight_layout(rect=[0,0,1,0.97])
    if save_path: plt.savefig(save_path, bbox_inches="tight")
    plt.show()

plot_training_curves(history, cfg, save_path=os.path.join(OUT_DIR, "fig_training_curves.png"))



In [ ]:
def plot_trajectory_plate(snapshot, cfg, example_idx=0, save_path=None):
    gt   = snapshot["gt"][example_idx].permute(1,2,0).squeeze().numpy()
    rec  = snapshot["recon"][example_idx].permute(1,2,0).squeeze().numpy()
    unc  = snapshot["uncertainty"][example_idx].squeeze().numpy()
    coords = torch.stack(snapshot["coords_trace"],0)[:, example_idx, :].numpy()
    K = coords.shape[0]; P, G = cfg.patch_size, cfg.grid_size
    cmap = "gray" if cfg.channels == 1 else None
    n_panels = K + 5
    fig = plt.figure(figsize=(2.1*n_panels, 2.6), dpi=200)
    gs = gridspec.GridSpec(1, n_panels, figure=fig, wspace=0.08)
    canvas = np.zeros_like(gt)
    for t in range(K):
        ax = fig.add_subplot(gs[0, t])
        r, c = int(coords[t,0]), int(coords[t,1])
        rs, re, cs, ce = r*P, (r+1)*P, c*P, (c+1)*P
        if gt.ndim == 2: canvas[rs:re, cs:ce] = gt[rs:re, cs:ce]
        else:            canvas[rs:re, cs:ce, :] = gt[rs:re, cs:ce, :]
        ax.imshow(canvas, cmap=cmap)
        ax.add_patch(plt.Rectangle((c*P-0.5, r*P-0.5), P, P, lw=2, ec="red", fc="none"))
        ax.set_title(f"Step {t+1}\n({r},{c})", fontsize=8, weight="bold"); ax.axis("off")
    
    ax = fig.add_subplot(gs[0, K])
    ax.imshow(gt, cmap=cmap, alpha=0.35)
    cy = coords[:,0]*P + P/2; cx = coords[:,1]*P + P/2
    ax.plot(cx, cy, "-", color="lime", lw=2, zorder=1)
    ax.scatter(cx, cy, c=range(K), cmap="winter", s=70, ec="black", zorder=2)
    for i,(x,y) in enumerate(zip(cx,cy)): ax.text(x+1.5, y-1.5, str(i+1), color="yellow", fontsize=8, weight="bold")
    ax.set_title("Trajectory", fontsize=9, weight="bold"); ax.axis("off")
    
    ax = fig.add_subplot(gs[0, K+1]); ax.imshow(canvas, cmap=cmap)
    ax.set_title("Model Sees", fontsize=9, weight="bold", color="navy"); ax.axis("off")
    ax = fig.add_subplot(gs[0, K+2]); ax.imshow(rec, cmap=cmap)
    ax.set_title("Reconstruction", fontsize=9, weight="bold"); ax.axis("off")
    ax = fig.add_subplot(gs[0, K+3]); ax.imshow(gt, cmap=cmap)
    ax.set_title("Ground Truth", fontsize=9, weight="bold"); ax.axis("off")
    ax = fig.add_subplot(gs[0, K+4]); im = ax.imshow(unc, cmap="inferno")
    ax.set_title("Uncertainty", fontsize=9, weight="bold"); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f"Epoch {snapshot['epoch']} â€” {cfg.dataset_name.upper()} "
                 f"(K={K}, visibleâ‰ˆ{K/cfg.num_patches*100:.0f}%)", weight="bold", y=1.04, fontsize=11)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, bbox_inches="tight")
    plt.show()

print(f"\n--- Trajectory plates ({len(snapshots)} snapshots) ---")
for snap in snapshots:
    plot_trajectory_plate(snap, cfg, example_idx=0,
        save_path=os.path.join(OUT_DIR, f"fig_trajectory_epoch{snap['epoch']}.png"))



In [ ]:
@torch.no_grad()
def capture_fresh(model, loader, cfg, n=6, deterministic=True):
    model.eval()
    images, _ = next(iter(loader))
    images = images[:n].to(DEVICE)
    raw01 = denormalize(images, cfg)
    out = run_episode(model, raw01, cfg, deterministic=deterministic, random_start=True)
    return dict(gt=raw01.cpu(), recon=out["recon01"].cpu(),
                uncertainty=out["uncertainty"].cpu(),
                coords=torch.stack(out["coords_trace"], 0).cpu())

def plot_full_plate(data, cfg, ex, save_path=None):
    gt = data["gt"][ex].permute(1,2,0).squeeze().numpy()
    rec = data["recon"][ex].permute(1,2,0).squeeze().numpy()
    unc = data["uncertainty"][ex].squeeze().numpy()
    coords = data["coords"][:, ex, :].numpy()
    K = coords.shape[0]; P, G = cfg.patch_size, cfg.grid_size
    cmap = "gray" if cfg.channels == 1 else None
    seen = np.zeros_like(gt)
    for t in range(K):
        r,c = int(coords[t,0]), int(coords[t,1])
        rs,re,cs,ce = r*P,(r+1)*P,c*P,(c+1)*P
        if gt.ndim==2: seen[rs:re,cs:ce] = gt[rs:re,cs:ce]
        else:          seen[rs:re,cs:ce,:] = gt[rs:re,cs:ce,:]
    n = K + 5
    fig = plt.figure(figsize=(2.1*n, 2.6), dpi=200)
    gs = gridspec.GridSpec(1, n, figure=fig, wspace=0.08)
    canvas = np.zeros_like(gt)
    for t in range(K):
        ax = fig.add_subplot(gs[0, t])
        r,c = int(coords[t,0]), int(coords[t,1])
        rs,re,cs,ce = r*P,(r+1)*P,c*P,(c+1)*P
        if gt.ndim==2: canvas[rs:re,cs:ce] = gt[rs:re,cs:ce]
        else:          canvas[rs:re,cs:ce,:] = gt[rs:re,cs:ce,:]
        ax.imshow(canvas, cmap=cmap)
        ax.add_patch(plt.Rectangle((c*P-0.5, r*P-0.5), P, P, lw=2, ec="red", fc="none"))
        ax.set_title(f"Step {t+1}\n({r},{c})", fontsize=8, weight="bold"); ax.axis("off")
    ax = fig.add_subplot(gs[0, K]); ax.imshow(gt, cmap=cmap, alpha=0.35)
    cy, cx = coords[:,0]*P+P/2, coords[:,1]*P+P/2
    ax.plot(cx, cy, "-", color="lime", lw=2, zorder=1)
    ax.scatter(cx, cy, c=range(K), cmap="winter", s=70, ec="black", zorder=2)
    for i,(x,y) in enumerate(zip(cx,cy)): ax.text(x+1.5, y-1.5, str(i+1), color="yellow", fontsize=8, weight="bold")
    ax.set_title("Trajectory", fontsize=9, weight="bold"); ax.axis("off")
    ax = fig.add_subplot(gs[0, K+1]); ax.imshow(seen, cmap=cmap)
    ax.set_title("Model Sees", fontsize=9, weight="bold", color="navy"); ax.axis("off")
    ax = fig.add_subplot(gs[0, K+2]); ax.imshow(rec, cmap=cmap)
    ax.set_title("Reconstruction", fontsize=9, weight="bold"); ax.axis("off")
    ax = fig.add_subplot(gs[0, K+3]); ax.imshow(gt, cmap=cmap)
    ax.set_title("Ground Truth", fontsize=9, weight="bold"); ax.axis("off")
    ax = fig.add_subplot(gs[0, K+4]); im = ax.imshow(unc, cmap="inferno")
    ax.set_title("Uncertainty", fontsize=9, weight="bold"); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f"{cfg.dataset_name.upper()} â€” K={K} visible â‰ˆ {K/cfg.num_patches*100:.0f}%",
                 weight="bold", y=1.04, fontsize=11)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, bbox_inches="tight")
    plt.show()

N_EX = 6
fresh = capture_fresh(model, val_loader, cfg, n=N_EX, deterministic=True)
for i in range(N_EX):
    plot_full_plate(fresh, cfg, i, save_path=os.path.join(OUT_DIR, f"fig_full_plate_ex{i}.png"))



In [ ]:
@torch.no_grad()
def run_random_episode(model, images_raw01, cfg):
    B = images_raw01.size(0)
    perm = torch.stack([torch.randperm(cfg.num_patches, device=DEVICE)[:cfg.budget] for _ in range(B)])
    embs_h, coords_h, steps_h = [], [], []
    for t in range(cfg.budget):
        a = perm[:, t]; row, col = action_to_coords(a, cfg)
        p = extract_single_patch(images_raw01, row, col, cfg)
        embs_h.append(model.encoder(p).unsqueeze(1))
        coords_h.append(torch.stack([row, col], dim=-1).unsqueeze(1))
        steps_h.append(torch.full((B,1), t, dtype=torch.long, device=DEVICE))
    mem = model.memory(torch.cat(embs_h,1), torch.cat(coords_h,1), torch.cat(steps_h,1))
    rp, _ = model.decoder(mem)
    return torch.clamp((rp+1)/2, 0, 1)

@torch.no_grad()
def compare_learned_vs_random(model, loader, cfg, n_batches=20, save_path=None):
    model.eval(); lp, rp = [], []
    for i, (images, _) in enumerate(loader):
        if i >= n_batches: break
        images = images.to(DEVICE); raw01 = denormalize(images, cfg)
        out = run_episode(model, raw01, cfg, deterministic=True, random_start=True)
        lp.append(compute_psnr(out["recon01"], raw01))
        rp.append(compute_psnr(run_random_episode(model, raw01, cfg), raw01))
    lm, rm = float(np.mean(lp)), float(np.mean(rp))
    print(f"[Test] Learned PSNR = {lm:.2f} dB | Random PSNR = {rm:.2f} dB | Î” = {lm-rm:+.2f} dB")
    fig, ax = plt.subplots(figsize=(5.5, 4.5), dpi=200)
    bars = ax.bar(["Random\nSelection", "Learned\nPolicy"], [rm, lm],
                  color=[C_RANDOM, C_LEARN], width=0.55, edgecolor="black", lw=0.6)
    ax.set_ylabel("PSNR (dB)"); ax.set_title(f"Learned vs Random Patch Selection (K={cfg.budget})", weight="bold")
    for i, v in enumerate([rm, lm]): ax.text(i, v+0.15, f"{v:.2f} dB", ha="center", weight="bold")
    ax.set_ylim(0, max(lm, rm)*1.18); plt.tight_layout()
    if save_path: plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    return lm, rm

print("\n--- Learned vs Random ---")
learned_psnr, random_psnr = compare_learned_vs_random(
    model, val_loader, cfg, save_path=os.path.join(OUT_DIR, "fig_learned_vs_random.png"))



In [ ]:
@torch.no_grad()
def run_episode_with_K(model, images_raw01, cfg, K, deterministic=True):
    B = images_raw01.size(0)
    visited = make_initial_mask(B, cfg)
    embs_h, coords_h, steps_h = [], [], []
    action = torch.randint(0, cfg.num_patches, (B,), device=DEVICE)
    for t in range(K):
        row, col = action_to_coords(action, cfg)
        p = extract_single_patch(images_raw01, row, col, cfg)
        embs_h.append(model.encoder(p).unsqueeze(1))
        coords_h.append(torch.stack([row, col], dim=-1).unsqueeze(1))
        steps_h.append(torch.full((B,1), t, dtype=torch.long, device=DEVICE))
        mem = model.memory(torch.cat(embs_h,1), torch.cat(coords_h,1), torch.cat(steps_h,1))
        visited = update_mask(visited, action)
        if t < K-1:
            bf = torch.full((B,1), (cfg.budget - t)/cfg.budget, device=DEVICE)
            logits, _ = model.policy(mem, visited, bf)
            action, _, _ = ActiveExplorationPolicy.sample_action(logits, deterministic)
    rp, _ = model.decoder(mem)
    return torch.clamp((rp+1)/2, 0, 1)

@torch.no_grad()
def budget_scaling_curve(model, loader, cfg, n_batches=10, save_path=None):
    model.eval()
    budgets = list(range(1, cfg.budget+1))
    learned_means = []
    for K in budgets:
        ps = []
        for i, (images, _) in enumerate(loader):
            if i >= n_batches: break
            images = images.to(DEVICE); raw01 = denormalize(images, cfg)
            recon = run_episode_with_K(model, raw01, cfg, K, deterministic=True)
            ps.append(compute_psnr(recon, raw01))
        learned_means.append(float(np.mean(ps)))
        print(f"  K={K}: PSNR = {learned_means[-1]:.2f} dB")
    fig, ax = plt.subplots(figsize=(6.5, 4.5), dpi=200)
    ax.plot(budgets, learned_means, "o-", color=C_LEARN, lw=2, markersize=8, label="Learned policy")
    ax.axvline(cfg.budget, color="gray", ls=":", lw=1)
    ax.set_xlabel("Observation Budget K (steps)")
    ax.set_ylabel("PSNR (dB)")
    ax.set_title("Reconstruction Quality vs Observation Budget", weight="bold")
    ax.set_xticks(budgets)
    ax.legend()
    plt.tight_layout()
    if save_path: plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    return dict(budgets=budgets, psnr=learned_means)

print("\n--- Budget scaling curve (K = 1..%d) ---" % cfg.budget)
budget_scaling_curve(model, val_loader, cfg,
                     save_path=os.path.join(OUT_DIR, "fig_budget_scaling.png"))

print("\n" + "="*70)
print("âœ“ All figures saved to:", OUT_DIR)
print("="*70)
print("  fig_samples_and_grid.png   â€” dataset + patch grid")
print("  fig_architecture.png        â€” model architecture")
print("  fig_training_curves.png     â€” 2x3 training curves")
print("  fig_trajectory_epoch*.png   â€” trajectory plates per snapshot")
print("  fig_full_plate_ex*.png      â€” per-example reconstruction plates")
print("  fig_learned_vs_random.png   â€” learned vs random comparison")
print("  fig_budget_scaling.png      â€” PSNR vs K")